In [1]:
# Installing the dependencies
import random
import time
from datetime import datetime

In [2]:
gateways = ["Stripe", "PayPal", "BankAPI"]
statuses = ["Success", "Failed"]

### Creating the Payment Data Generator

In [6]:
import random
import time
from datetime import datetime

gateways = ["Stripe", "PayPal", "BankAPI"]
statuses = ["Success", "Failed"]

def generate_payment():
    return {
        "payment_id": random.randint(1000,9999),
        "gateway": random.choice(gateways),
        "amount": round(random.uniform(10,500),2),
        "status": random.choices(
            statuses,
            weights=[0.9,0.1]
        )[0],
        "response_time": random.randint(100,5000),
        "timestamp": datetime.now().strftime("%H:%M:%S")
    }

for _ in range(30):
    print(generate_payment())
    time.sleep(1)

{'payment_id': 4955, 'gateway': 'BankAPI', 'amount': 293.99, 'status': 'Success', 'response_time': 2446, 'timestamp': '14:56:02'}
{'payment_id': 4549, 'gateway': 'Stripe', 'amount': 74.01, 'status': 'Success', 'response_time': 2508, 'timestamp': '14:56:03'}
{'payment_id': 8507, 'gateway': 'Stripe', 'amount': 227.21, 'status': 'Success', 'response_time': 4952, 'timestamp': '14:56:04'}
{'payment_id': 9335, 'gateway': 'PayPal', 'amount': 143.91, 'status': 'Success', 'response_time': 4486, 'timestamp': '14:56:05'}
{'payment_id': 4077, 'gateway': 'BankAPI', 'amount': 300.27, 'status': 'Success', 'response_time': 3196, 'timestamp': '14:56:06'}
{'payment_id': 1610, 'gateway': 'Stripe', 'amount': 445.02, 'status': 'Success', 'response_time': 2453, 'timestamp': '14:56:07'}
{'payment_id': 7849, 'gateway': 'Stripe', 'amount': 116.92, 'status': 'Failed', 'response_time': 3071, 'timestamp': '14:56:08'}
{'payment_id': 4414, 'gateway': 'Stripe', 'amount': 161.1, 'status': 'Success', 'response_time': 

In [5]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
import random
from datetime import datetime

### Creating a Streaming Data Function

In [7]:
def generate_payments(n=30):
    gateways = ["Stripe", "PayPal", "BankAPI"]
    
    for _ in range(n):
        yield {
            "payment_id": random.randint(1000,9999),
            "gateway": random.choice(gateways),
            "status": random.choices(
                ["Success", "Failed"],
                weights=[0.85, 0.15]
            )[0],
            "timestamp": datetime.now().strftime("%H:%M:%S")
        }

### Building the Apache Beam Pipeline

In [8]:
def detect_failures(element):
    if element["status"] == "Failed":
        print("Payment Failure Detected:", element)
    return element

with beam.Pipeline(options=PipelineOptions()) as pipeline:
    (
        pipeline
        | "Create Payments" >> beam.Create(generate_payments(30))
        | "Detect Failures" >> beam.Map(detect_failures)
    )

Payment Failure Detected: {'payment_id': 1747, 'gateway': 'PayPal', 'status': 'Failed', 'timestamp': '15:02:37'}
Payment Failure Detected: {'payment_id': 4249, 'gateway': 'BankAPI', 'status': 'Failed', 'timestamp': '15:02:37'}
Payment Failure Detected: {'payment_id': 9119, 'gateway': 'BankAPI', 'status': 'Failed', 'timestamp': '15:02:37'}


### Building the Detect Failure Rate (Real Payment Monitoring) system

In [9]:
def tag_failures(element):
    if element["status"] == "Failed":
        return("failed", 1)
    else:
        return("success", 1)

### Calculating the Failure Rate

In [11]:
def calculate_failure_rate(counts):
    counts_dict = dict(counts)
    
    failed = counts_dict.get("failed", 0)
    success = counts_dict.get("success", 0)
    
    total = failed + success
    
    if total > 0:
        failure_rate = failed / total
        
        print(f"Total: {total}, Failed: {failed}, Failure Rate: {failure_rate:.2f}")
        
        if failure_rate > 0.2: # if > 20%
            print("⚠️ ALERT: High Payment Failure Rate Detected!")
    
    return counts

In [12]:
### Updating the Apache Beam Pipeline
with beam.Pipeline(options=PipelineOptions()) as pipeline:
    
    (
        pipeline
        | "Create Payments" >> beam.Create(generate_payments(50))
        | "Tag Failures" >> beam.Map(tag_failures)
        | "Count" >> beam.CombinePerKey(sum)
        | "Collect" >> beam.combiners.ToList()
        | "Calculate Failure Rate" >> beam.Map(calculate_failure_rate)
    )

Total: 50, Failed: 3, Failure Rate: 0.06


### Detecting Failures Per Gateway (Real Payment Failover Logic)

In [13]:
def tag_gateway_failures(element):
    gateway = element["gateway"]
    
    if element["status"] == "Failed":
        return ((gateway, "failed"), 1)
    else:
        return ((gateway, "success"), 1)

### Caluclating Gateway Failure Rate

In [14]:
def calculate_gateway_failure_rate(elements):
    
    gateway_stats = {}
    
    for (gateway, status), count in elements:
        
        if gateway not in gateway_stats:
            gateway_stats[gateway] = {"failed": 0, "success": 0}
        
        gateway_stats[gateway][status] += count
    
    for gateway, stats in gateway_stats.items():
        
        failed = stats["failed"]
        success = stats["success"]
        total = failed + success
        
        if total > 0:
            failure_rate = failed / total
            
            print(f"{gateway} → Failure Rate: {failure_rate:.2f}")
            
            if failure_rate > 0.25:
                print(f"⚠️ FAILOVER ALERT: Switch from {gateway}")
    
    return elements

### Now, again updating the Apache Beam Pipeline

In [15]:
with beam.Pipeline(options=PipelineOptions()) as pipeline:
    
    (
        pipeline
        | "Create Payments" >> beam.Create(generate_payments(50))
        | "Tag Gateway Failures" >> beam.Map(tag_gateway_failures)
        | "Count Gateway Failures" >> beam.CombinePerKey(sum)
        | "Collect Results" >> beam.combiners.ToList()
        | "Calculate Gateway Failure Rate" >> beam.Map(calculate_gateway_failure_rate)
    )

BankAPI → Failure Rate: 0.22
Stripe → Failure Rate: 0.06
PayPal → Failure Rate: 0.07


### Using a 1 minute sliding or fixed window to compute failure rates continuously

In [22]:
with beam.Pipeline(options=PipelineOptions()) as pipeline:
    (
        pipeline
        | "Create Payments" >> beam.Create(generate_payments(100))
        | "Add Timestamps" >> beam.Map(lambda x: beam.window.TimestampedValue(x, time.time()))
        | "Fixed 1-Minute Window" >> beam.WindowInto(window.FixedWindows(60))
        | "Tag Gateway Failures" >> beam.Map(tag_gateway_failures)
        | "Count Gateway Failures Per Key" >> beam.CombinePerKey(sum)
        | "Log Gateway Failures" >> beam.Map(lambda kv: print(f"Gateway: {kv[0]}, Failures: {kv[1]}"))
    )

Gateway: ('Stripe', 'success'), Failures: 31
Gateway: ('BankAPI', 'failed'), Failures: 7
Gateway: ('PayPal', 'success'), Failures: 24
Gateway: ('BankAPI', 'success'), Failures: 24
Gateway: ('PayPal', 'failed'), Failures: 7
Gateway: ('Stripe', 'failed'), Failures: 7


### Calculating Failure Rate per gateway

In [23]:
def calculate_failure_rate_per_gateway(elements):
    gateway_stats = {}
    for (gateway, status), count in elements:
        if gateway not in gateway_stats:
            gateway_stats[gateway] = {"failed": 0, "success": 0}
        gateway_stats[gateway][status] += count
    
    for gateway, stats in gateway_stats.items():
        total = stats["failed"] + stats["success"]
        if total > 0:
            failure_rate = stats["failed"] / total
            print(f"{gateway} → Failure Rate: {failure_rate:.2f}")
            if failure_rate > 0.25:
                print(f"⚠️ FAILOVER ALERT: Consider switching from {gateway}")
    return elements

In [28]:
import apache_beam as beam
import time
import random
import apache_beam.transforms.window as window
from apache_beam.options.pipeline_options import PipelineOptions

with beam.Pipeline(options=PipelineOptions()) as pipeline:
    
    (
        pipeline
        | "Create Payments" >> beam.Create(generate_payments(100))

        | "Log Payments" >> beam.Map(log_payments)

        | "Simulate Failover" >> beam.Map(simulate_failover)

        | "Add Timestamp" >> beam.Map(
            lambda x: beam.window.TimestampedValue(x, time.time())
        )

        | "Window" >> beam.WindowInto(window.FixedWindows(60))

        | "Tag Failures" >> beam.Map(tag_gateway_failures)

        | "Count Failures" >> beam.CombinePerKey(sum)

        # FIX HERE
        | "Collect" >> beam.combiners.ToList().without_defaults()

        | "Calculate Failure Rate" >> beam.Map(
            calculate_failure_rate_per_gateway
        )
    )

BankAPI → Failure Rate: 0.12
Stripe → Failure Rate: 0.21
PayPal → Failure Rate: 0.17


In [36]:
import plotly.graph_objects as go

gateways = ["Stripe", "PayPal", "BankAPI"]
failure_rates = [0.21, 0.17, 0.12]

fig = go.Figure(
    data=[
        go.Bar(
            x=gateways,
            y=failure_rates,
            text=[f"{rate:.2f}" for rate in failure_rates],
            textposition='auto'
        )
    ]
)

fig.update_layout(
    title="Real-Time Payment Gateway Failure Rates",
    xaxis_title="Payment Gateways",
    yaxis_title="Failure Rate",
    template="plotly_dark"
)

fig.show()